# 02 — Evidence Layer: Provenance, Parsing, Chunking

*Notebook 2 of 4. Continues from `01_llm_foundations.ipynb`.*

Before we can retrieve anything, we need documents that are (a) real, (b) machine-readable, and (c) traceable back to their source. This notebook builds that evidence layer: define the corpus, acquire the official PDFs, parse them into text, and chunk that text into retrievable units.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data/corpus_manifest.json').exists():
            return candidate
    raise FileNotFoundError(
        'Could not find the workshop repo root (looked for data/corpus_manifest.json in this '
        'directory and its parents). Run this notebook from inside Gravitas_Workshop_Starter.'
    )


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Workshop root:', ROOT)

from config import load_workshop_env
load_workshop_env()  # load .env once, up front, so later @observe spans don't warn about missing keys

from observability.tracing import flush_langfuse, dashboard_base_url


## Mission 3 — Define the corpus

### 🧠 Concept: RAG starts with a *library*, not with a vector database

A **corpus** is the set of documents your system is allowed to search. For this workshop that's official investor documents for TCS, Infosys and HCLTech — three companies chosen to keep the corpus small enough to fully process live; the same approach scales to any set of official filings.

Each piece of evidence must preserve **provenance**: company, period, document type, filename, page and chunk ID. Otherwise a good-looking answer becomes impossible to audit.


In [ ]:
import json
manifest = json.loads((ROOT / 'data/corpus_manifest.json').read_text())
[(x['company'], x['doc_type'], x['period'], x['filename']) for x in manifest]


### Acquire the official PDFs

Run this from a terminal (not a notebook cell) so you can watch download progress:

```bash
uv run python scripts/fetch_documents.py
```

In [ ]:
docs = sorted((ROOT / 'data/documents').glob('*.pdf'))
[(p.name, round(p.stat().st_size / 1_000_000, 2)) for p in docs]


## Mission 4 — Parsing: turn page layouts into evidence text

### 🧠 Concept

A PDF is a **visual page format**, not a clean text database. Financial filings may contain columns, tables, repeated headers, scanned pages and page-number mismatches.

Parsing is the stage where we ask:

> Did the evidence survive conversion into machine-readable text?

We will first test a short Infosys release before processing everything.


📎 **Script reference:** `retrieval/ingestion/parsing.py` (`parse_pdf_pages`) — a thin wrapper around LiteParse. Run it standalone with `uv run python -m retrieval.ingestion.parsing` to parse the first document in `data/documents/` from a terminal.


In [ ]:
from retrieval.ingestion.parsing import parse_pdf_pages

sample = ROOT / 'data/documents/Infosys_Q4_FY25_Press_Release.pdf'
pages = parse_pdf_pages(sample)
print('Parsed pages:', len(pages))


In [ ]:
# ✍️ YOUR TURN 1 — Inspect the parser instead of trusting it blindly.
page_to_inspect = 0      # TODO: try another page after your first run
search_term = 'revenue'  # TODO: try 'margin', 'guidance' or another finance term

text = pages[page_to_inspect]['text']
print(text[:2200])
print('\nContains search term?', search_term.lower() in text.lower())


### 🔎 What to inspect

Look for one example of each:

- a number that survived correctly,
- a heading or sentence boundary that survived,
- a layout/table artifact that became messy.

If parsing destroys the evidence, no retrieval algorithm later can magically recover it.


### ✍️ YOUR TURN 2 — Inspect more than the first page
A parser can succeed on page 1 and silently fail later. Write a tiny helper that finds pages containing a finance term, then inspect one of those pages manually.


In [ ]:
def pages_with_term(parsed_pages, term):
    # TODO: return page numbers containing term case-insensitively
    return []
pages_with_term(pages, 'margin')[:10]


## Mission 5 — Chunking: decide what becomes retrievable

### 🧠 Concept

We cannot send hundreds of pages to the model for every question. So we split parsed pages into smaller evidence units called **chunks**.

- **Too small:** a number may be separated from the sentence that explains it.
- **Too large:** retrieval gets noisy and context becomes expensive.
- **Overlap:** repeats a little text across boundaries so important sentences are less likely to be cut in half.

<img src="../assets/notebook/chunking.png" width="900" alt="Workshop slide illustrating chunk size and overlap">


📎 **Script reference:** `retrieval/ingestion/chunking.py` (`chunk_pages`) — wraps Agno's `RecursiveChunking`. Run it standalone with `uv run python -m retrieval.ingestion.chunking` to chunk a small sample and print the result.


In [ ]:
from retrieval.ingestion.chunking import chunk_pages

# ✍️ YOUR TURN 3 — These are design choices, not magic constants.
CHUNK_SIZE = 1200  # TODO: after the first run, try 700 or 1700
OVERLAP = 120      # TODO: after the first run, try 0 or ~10% of chunk size

chunks = chunk_pages(pages, chunk_size=CHUNK_SIZE, overlap=OVERLAP)
print('Chunks:', len(chunks))
[(c['page'], c['chunk'], c['text'][:220].replace('\n',' ')) for c in chunks[:5]]


### 🔮 Mini experiment

Change **one parameter at a time** and rerun the previous cell.

Predict first:

- If chunk size gets smaller, will the number of chunks go **up or down**?
- If overlap gets larger, will repeated text go **up or down**?
- Which setting looks easier to cite and understand?

There is no universally correct chunk size; the right answer depends on the documents and questions.


---
**Next:** open `03_indexing_and_rag.ipynb` to turn this corpus into a vector index and close the first RAG loop.
